In [0]:
storage_path = "abfss://bronze@stedukronretaildl.dfs.core.windows.net/"

display(dbutils.fs.ls(storage_path))

In [0]:
csv_path = storage_path + "raw/blob/products.csv"

df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(csv_path)

display(df.limit(10))

In [0]:
json_path = storage_path + "raw/api/campaigns.json"

print(dbutils.fs.head(json_path, 1000))

In [0]:

from pyspark.sql.functions import current_timestamp, col
from pathlib import PurePosixPath

# ADLS Gen2 paths
source_base = (
    "abfss://bronze@stedukronretaildl.dfs.core.windows.net/raw"
)

bronze_base = (
    "abfss://bronze@stedukronretaildl.dfs.core.windows.net/processed"
)

# Source folders and formats
sources = {
    "blob": "csv",
    "api": "json",
    "sql": "json",
    "adls": "parquet"
}

def ingest_to_bronze(source_name, file_format):

    source_path = f"{source_base}/{source_name}"
    files = dbutils.fs.ls(source_path)

    for file in files:

        if file.isDir():
            continue

        file_name = file.name
        table_name = PurePosixPath(file_name).stem

        print(f"Processing: {source_name}/{file_name}")

        # Read source data
        if file_format == "csv":
            df = (
                spark.read
                .option("header", True)
                .option("inferSchema", True)
                .csv(file.path)
            )

        elif file_format == "json":
            df = (
                spark.read
                .option("multiLine", True)
                .option("inferSchema", True)
                .json(file.path)
            )

        elif file_format == "parquet":
            df = spark.read.parquet(file.path)

        # Add ingestion metadata

        df = (
           df.withColumn("_ingested_at", current_timestamp())
           .withColumn("_source_file", col("_metadata.file_path"))
)

        # Write as Delta
        output_path = (
            f"{bronze_base}/{source_name}/{table_name}"
        )

        (
            df.write
            .format("delta")
            .mode("overwrite")
            .save(output_path)
        )

        print(f"Successfully written: {output_path}")
        
        df.unpersist()


# Process all four source folders
for source_name, file_format in sources.items():
    ingest_to_bronze(source_name, file_format)

print("Bronze ingestion completed!")


In [0]:
from collections import defaultdict

bronze_path = (
    "abfss://bronze@stedukronretaildl.dfs.core.windows.net/processed"
)

results = defaultdict(list)

for source in ["blob", "api", "sql", "adls"]:
    source_path = f"{bronze_path}/{source}"

    datasets = dbutils.fs.ls(source_path)

    for dataset in datasets:
        df = spark.read.format("delta").load(dataset.path)

        results[source].append({
            "dataset": dataset.name.rstrip("/"),
            "row_count": df.count(),
            "columns": len(df.columns)
        })

for source, datasets in results.items():
    print(f"\n{source.upper()} — {len(datasets)} datasets")

    for dataset in datasets:
        print(
            f"{dataset['dataset']}: "
            f"{dataset['row_count']} rows, "
            f"{dataset['columns']} columns"
        )